# Location set covering

Choose the smallest candidate facility set that reaches every demand location within a stated service threshold. Unlike maximal covering, set covering cannot leave demand uncovered.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'implementations/python/set-covering'))
sys.path.insert(0, str(Path.cwd().parent / 'implementations/python/maximal-covering'))
import numpy as np
import matplotlib.pyplot as plt
from set_covering import solve_set_cover
from maximal_covering import solve_maximal_covering


## 1. A small hypothetical service area

Rows are demand locations and columns are candidate sites. Costs are illustrative travel-time minutes.

In [ ]:
costs = np.array([[4, 18, 22, 12], [7, 9, 19, 13], [16, 5, 11, 17], [20, 8, 9, 7], [12, 16, 4, 18], [17, 11, 8, 5]])
threshold = 10
sites, count, assignments = solve_set_cover(costs, threshold)
print('Minimum covering set:', sites)
print('Facilities required:', count)
print('Assignments:', assignments)


## 2. Inspect eligible facilities

A cell is eligible when its cost is no greater than the service threshold. Every row must contain at least one selected eligible candidate.

In [ ]:
eligible = costs <= threshold
fig, ax = plt.subplots(figsize=(8,4))
ax.imshow(eligible, aspect='auto', cmap='Greys', vmin=0, vmax=1)
ax.set_xticks(range(costs.shape[1]), [f'Site {j}' for j in range(costs.shape[1])])
ax.set_yticks(range(costs.shape[0]), [f'Demand {i}' for i in range(costs.shape[0])])
ax.set_title(f'Eligible set-cover relations within {threshold} minutes')
ax.set_xlabel('Candidate facilities'); ax.set_ylabel('Demand locations')
for i in range(costs.shape[0]):
    for j in range(costs.shape[1]): ax.text(j,i,'✓' if eligible[i,j] else '·',ha='center',va='center')
plt.tight_layout(); plt.show()


## 3. Compare with maximal covering

Set covering minimizes the number of facilities needed to reach everyone. Maximal covering fixes an upper budget and maximizes the demand reached.

In [ ]:
weights = np.array([1200, 800, 950, 700, 1100, 650])
for budget in (1, 2, 3):
    chosen, covered, flags = solve_maximal_covering(costs, budget, threshold, demand_weights=weights)
    print(f'MCLP budget={budget}: sites={chosen}, covered={covered:.0f}/{weights.sum()}')
print(f'Set covering: sites={sites}, count={count}, all demand covered')


## 4. Feasibility and limitations

If a demand row has no candidate within the threshold, the model is infeasible. This baseline has equal opening costs and unlimited facility capacity; it does not model queues, outages or backup coverage.